In [0]:
#Step 1st: Importing required librarieslike functions, deltatable, datetime, uuid.

from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
#Catalog usage initiation.

spark.sql("Use catalog data_novacart_databricks")

In [0]:
#Schema creation for bronze_schema.

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze_schema")

In [0]:
#Step 2nd: Creation of Bronze control table that can hold water mark for each source table.
#It will help pipeline remember : latest timestamp(ts) already processed, latest primarykey(pk) processed at that timestamp,how many rows were written in the latest run. This will make it incremental and re-run safe.

spark.sql(""" 
        
create table if not exists data_novacart_databricks.bronze_schema.ingestion_control(
    layer string,
    table_name string,
    ts_col string,
    pk_col string,
    last_successful_ts timestamp,
    last_successful_pk bigint,
    last_run_id string,
    rows_written bigint,
    run_status string,
    updated_at timestamp
    
)   
using delta
"""

)
 

In [0]:
# Step 3rd: Source table configuration 
# Here we are going to define which table will be loaded to bronze and which column should be used as pk and ts/watermark column and it will also create a bronze_run_id for the current pipeline run.

tables_config = {
    "orders" : {"pk_col" : "order_id" , "ts_col" : "updated_at"},
    "products" : {"pk_col" : "product_id" , "ts_col" : "updated_at"},
    "payments" : {"pk_col" : "payment_id" , "ts_col" : "processed_at"}
}

bronze_run_id = str(uuid.uuid4())
print("Current bronze run id",bronze_run_id)

In [0]:
#Step 4th: Helper function creation
#This will contin 2 reusable functions: get_last_successful_watermark(), upsert_bronze_control(). first will read the last processed watermark from the control table and second would update the cntrol table after the successfull bronze load.

#These function will keep the main logic cleaner and easier to understand.

def get_last_successful_watermark(table_name: str):
    ctrl = (
        spark.table("data_novacart_databricks.bronze_schema.ingestion_control")
        .filter(
            (f.col("layer") == "bronze") &
            (f.col("table_name") == table_name) &
            (f.col("run_status") == "success")
        )
        .orderBy(f.col("updated_at").desc())
        .limit(1)
    )

    rows = ctrl.collect()

    if not rows:
        return None, None

    return rows[0]["last_successful_ts"], rows[0]["last_successful_pk"]

In [0]:
def upsert_bronze_control(table_name, ts_col, pk_col, last_ts, last_pk, rows_written, run_id):
    control_df = spark.createDataFrame(
        [(
            "bronze",
            table_name,
            ts_col,
            pk_col,
            last_ts,
            int(last_pk) if last_pk is not None else None,
            run_id,
            int(rows_written),
            "success",
            datetime.utcnow()
        )],
        schema="""
        layer string,
        table_name string,
        ts_col string,
        pk_col string,
        last_successful_ts timestamp,
        last_successful_pk bigint,
        last_run_id string,
        rows_written bigint,
        run_status string,
        updated_at timestamp
        """
    )

    dt = DeltaTable.forName(spark, "data_novacart_databricks.bronze_schema.ingestion_control")
    dt.alias("t").merge(
        control_df.alias("s"),
        "t.table_name = s.table_name AND t.layer = s.layer"
    ).whenMatchedUpdate(set={
        "ts_col": "s.ts_col",
        "pk_col": "s.pk_col",
        "last_successful_ts": "s.last_successful_ts",
        "last_successful_pk": "s.last_successful_pk",
        "last_run_id": "s.last_run_id",
        "rows_written": "s.rows_written",
        "run_status": "s.run_status",
        "updated_at": "s.updated_at"
    }).whenNotMatchedInsertAll().execute()

In [0]:
#Step 5th: Bronze incremental load loop
##This is the main logic of the bronze pipeline. For each table the notebook reads the last water mark, reads the source SQL table, filters only new change/rows , adds bronze audit columns , appends the row into Bronzde Delta table and updates the control table with the latest watermark. This is the core Incremental Loading Logic.

f = F

for table_name, cfg in tables_config.items():
    ts_col = cfg["ts_col"]
    pk_col = cfg["pk_col"]
    
    source_table = f"`data_novacart_connection_catalog`.dbo.{table_name}"
    target_table = f"data_novacart_databricks.bronze_schema.{table_name}_raw"
    
    last_successful_ts, last_successful_pk = get_last_successful_watermark(table_name)
    print(f"\n=== Processing {table_name} ===")
    print(f"Last successful ts: {last_successful_ts}")
    print(f"Last successful pk: {last_successful_pk}")

    source_df = spark.read.table(source_table).withColumn(ts_col, F.col(ts_col).cast("timestamp"))
    if last_successful_ts is None:
        rows_to_load = source_df
    else:
        rows_to_load = source_df.filter(
            (F.col(ts_col) > F.lit(last_successful_ts)) |
            (
                (F.col(ts_col) == F.lit(last_successful_ts)) &
                (F.col(pk_col).cast("long") > F.lit(last_successful_pk))
            )
        )

    rows_to_load = (
        rows_to_load
        .withColumn("bronze_ingested_at", F.current_timestamp())
        .withColumn("bronze_run_id", F.lit(bronze_run_id))
        .withColumn("bronze_source_table", F.lit(source_table))
    )

    rows_count = rows_to_load.count()
    print(f"{table_name} has {rows_count} rows to load")

    if rows_count == 0:
        print(f"No new row to be printed for {table_name}.")
        upsert_bronze_control(
            table_name,
            ts_col,
            pk_col,
            last_successful_ts,
            last_successful_pk,
            rows_count,
            bronze_run_id
        )
        continue
    
    rows_to_load.write.format("delta").mode("append").saveAsTable(target_table)
    max_ts = rows_to_load.agg(F.max(ts_col).alias("max_ts")).collect()[0]["max_ts"]
    max_pk = (
        rows_to_load
        .filter(F.col(ts_col) == F.lit(max_ts))
        .agg(F.max(F.col(pk_col).cast("long")).alias("max_pk"))
        .collect()[0]["max_pk"]
    )

    upsert_bronze_control(
        table_name,
        ts_col,
        pk_col,
        max_ts,
        max_pk,
        rows_count,
        bronze_run_id
    )
    print(f"Wrote {rows_count} rows to {target_table}")

In [0]:
#Step 6th: Quick Validation and Clean up
#This is the last step of the bronze pipeline. This final cell will print Bronze row count and display the control table so that you can verify that the incremental logic is working correctly. It will delete all the temp tables created during the run and delete the bronze_run_id from the control table.

print(f"Orders bronze count", spark.sql(f"select count(*) from data_novacart_databricks.bronze_schema.orders_raw").collect()[0][0])
print(f"Payments bronze count", spark.sql(f"select count(*) from data_novacart_databricks.bronze_schema.payments_raw").collect()[0][0])
print(f"Products bronze count", spark.sql(f"select count(*) from data_novacart_databricks.bronze_schema.products_raw").collect()[0][0])

display(spark.sql("select * from data_novacart_databricks.bronze_schema.ingestion_control").orderBy("table_name"))